In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [2]:
df = pd.read_csv('Churn_Modelling.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
#preprocess the data
#Drop irrelevant feature
df = df.drop(columns=['RowNumber','CustomerId','Surname'])

In [4]:
df.head(3)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1


In [5]:
#Encode categorical variables
label_encoder_gender = LabelEncoder()
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])

In [6]:
df.head(3)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1


In [7]:
# One hot encode Geography column
from sklearn.preprocessing import OneHotEncoder
ohe_geography =  OneHotEncoder(sparse_output=False)
geo_encoder = ohe_geography.fit_transform(df[['Geography']])
geo_encoder

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [8]:
ohe_geography.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [9]:
ohe_geography_df = pd.DataFrame(geo_encoder,columns = ohe_geography.get_feature_names_out())

In [10]:
ohe_geography_df.head(3)

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0


In [11]:
df = pd.concat([df.drop('Geography', axis=1), ohe_geography_df], axis=1)

In [12]:
df.head(3)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0


In [13]:
## Save the encoders and scalers
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(ohe_geography,file)

In [14]:
#Divide the dataset into independent and dependent feature
X = df.drop('Exited', axis=1)
y = df['Exited']

In [15]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [16]:
#Scale these features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

In [17]:
X_test = scaler.transform(X_test)

In [18]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler,file)

### ANN IMPLEMENTATION

In [19]:
import tensorflow as tf

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [21]:
X_train.shape

(8000, 12)

In [22]:
#Build ANN Model
model = Sequential(
    [
    Dense(64, activation='relu',input_shape = (X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [23]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [24]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss= tf.keras.losses.BinaryCrossentropy()

In [25]:
model.compile(optimizer= opt, loss = loss, metrics=['accuracy'])

In [26]:
#Compile the model
#model.compile(optimizer= "adam", loss = 'binary_crossentropy', metrics=['accuracy'])

In [27]:
##Setup the Tensorboard
log_dir = "logs/fit/"+ datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir = log_dir, histogram_freq=1)

In [28]:
#Set up early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=16,restore_best_weights=True)

In [29]:
## Train the model
history = model.fit(X_train,y_train, validation_data=(X_test,y_test),epochs =100,
                    callbacks = [tensorflow_callback,early_stopping_callback]
                    )

Epoch 1/100


250/250 [==============================] - 1s 3ms/step - loss: 0.4000 - accuracy: 0.8347 - val_loss: 0.3526 - val_accuracy: 0.8555
Epoch 2/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3538 - accuracy: 0.8562 - val_loss: 0.3445 - val_accuracy: 0.8585
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3483 - accuracy: 0.8585 - val_loss: 0.3434 - val_accuracy: 0.8560
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3421 - accuracy: 0.8620 - val_loss: 0.3485 - val_accuracy: 0.8550
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3393 - accuracy: 0.8611 - val_loss: 0.3468 - val_accuracy: 0.8525
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3391 - accuracy: 0.8635 - val_loss: 0.3384 - val_accuracy: 0.8580
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3369 - accuracy: 0.8605 - val_loss: 0.3416 - val_accuracy: 0.85

In [30]:
model.save('model.h5')

d:\D Drive Data\Github Projects\churn_ann_classification\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [31]:
##Load TensorBoard extension
%load_ext tensorboard

In [ ]:
## Run this in terminal
#tensorboard --logdir logs